# Consigna-exams-2025-DIA-II-jul-2026

**UASD - Ciencias de Datos I (INF-8237-C2)**  
**Quiz de recuperación 01 - 5 puntos**  
**Participante:** Marlenis Judith Concepción Cuevas  
**Profesor:** Dr. Silverio

Cuaderno reproducible para EDA, estadística inferencial y machine learning con OULAD.

## 1. Preparación del entorno

In [ ]:
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

try:
    from statsmodels.formula.api import ols
    import statsmodels.api as sm
    from statsmodels.multivariate.manova import MANOVA
except Exception:
    sm = None
    MANOVA = None

RANDOM_STATE = 8237
sns.set_theme(style="whitegrid", palette="deep")

## 2. Carga e integración del dataset OULAD

Sube `oulad.zip` al entorno de Colab o monta Google Drive y ajusta `DATA_PATH`.

In [ ]:
DATA_PATH = Path("oulad.zip")
if not DATA_PATH.exists():
    DATA_PATH = Path("/content/oulad.zip")

if not DATA_PATH.exists():
    raise FileNotFoundError("Sube oulad.zip a Colab o monta Google Drive y ajusta DATA_PATH.")

def read_zip_csv(archive, filename, **kwargs):
    match = [name for name in archive.namelist() if name.endswith(filename)][0]
    with archive.open(match) as f:
        return pd.read_csv(f, **kwargs)

with zipfile.ZipFile(DATA_PATH) as z:
    student_info = read_zip_csv(z, "studentInfo.csv")
    assessments = read_zip_csv(z, "assessments.csv")
    student_assessment = read_zip_csv(z, "studentAssessment.csv")
    student_vle_iter = read_zip_csv(z, "studentVle.csv", chunksize=500_000)
    partials = []
    for chunk in student_vle_iter:
        early = chunk.loc[chunk["date"].between(0, 28)]
        partials.append(
            early.groupby(["code_module", "code_presentation", "id_student"], as_index=False)
            .agg(clicks_28d=("sum_click", "sum"), dias_activos_28d=("date", "nunique"))
        )

clicks = pd.concat(partials, ignore_index=True).groupby(
    ["code_module", "code_presentation", "id_student"], as_index=False
).sum()

student_assessment["score"] = pd.to_numeric(student_assessment["score"], errors="coerce")
scores = (
    student_assessment
    .merge(assessments[["id_assessment", "code_module", "code_presentation"]], on="id_assessment", how="left")
    .groupby(["code_module", "code_presentation", "id_student"], as_index=False)
    .agg(promedio_evaluaciones=("score", "mean"), evaluaciones_entregadas=("id_assessment", "nunique"))
)

keys = ["code_module", "code_presentation", "id_student"]
df = student_info.merge(clicks, on=keys, how="left").merge(scores, on=keys, how="left")
df["aprobo"] = df["final_result"].isin(["Pass", "Distinction"]).astype(int)
df["resultado_ordinal"] = df["final_result"].map({"Withdrawn": 0, "Fail": 1, "Pass": 2, "Distinction": 3})
df["educacion_ordinal"] = df["highest_education"].map({
    "No Formal quals": 0, "Lower Than A Level": 1, "A Level or Equivalent": 2,
    "HE Qualification": 3, "Post Graduate Qualification": 4
})
df["edad_ordinal"] = df["age_band"].map({"0-35": 0, "35-55": 1, "55<=": 2})
df["score_pretest"] = (df["promedio_evaluaciones"] / 1.25).clip(0, 100)
df.head()

## 3. Estructura, faltantes y descriptivas

In [ ]:
display(df.info())
display(pd.DataFrame({"tipo": df.dtypes.astype(str), "faltantes": df.isna().sum(), "porcentaje": df.isna().mean() * 100}))

num_cols = ["clicks_28d", "dias_activos_28d", "promedio_evaluaciones", "score_pretest",
            "evaluaciones_entregadas", "num_of_prev_attempts", "studied_credits",
            "educacion_ordinal", "edad_ordinal", "aprobo"]
desc = df[num_cols].agg(["count", "mean", "median", "std", "var", "min", "max", stats.kurtosis]).T
desc["rango"] = desc["max"] - desc["min"]
display(desc)

display(df["final_result"].value_counts(normalize=False).rename("cantidad"))
display(pd.crosstab(df["gender"], df["final_result"], margins=True))
display(pd.crosstab(df["age_band"], df["final_result"], normalize="index") * 100)
display(df.groupby(["gender", "highest_education"])["promedio_evaluaciones"].agg(["count", "mean", "median", "std"]))

## 4. Visualizaciones, pivots y matriz correlacional

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
sns.histplot(df["promedio_evaluaciones"], kde=True, ax=axes[0, 0])
sns.boxplot(data=df, x="final_result", y="clicks_28d", ax=axes[0, 1])
sns.barplot(data=df, x="gender", y="aprobo", ax=axes[1, 0])
df["final_result"].value_counts().plot.pie(autopct="%1.1f%%", ax=axes[1, 1])
axes[1, 1].set_ylabel("")
plt.tight_layout()
plt.show()

corr_vars = ["clicks_28d", "dias_activos_28d", "promedio_evaluaciones", "score_pretest",
             "studied_credits", "num_of_prev_attempts", "educacion_ordinal", "edad_ordinal", "aprobo"]
plt.figure(figsize=(10, 7))
sns.heatmap(df[corr_vars].corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Matriz correlacional general")
plt.show()

## 5. Pruebas inferenciales: Pearson, Spearman, t-test, ANOVA, ANCOVA y MANOVA

In [ ]:
pares = [
    ("clicks_28d", "promedio_evaluaciones"),
    ("dias_activos_28d", "promedio_evaluaciones"),
    ("score_pretest", "promedio_evaluaciones"),
    ("clicks_28d", "aprobo"),
]
for x, y in pares:
    tmp = df[[x, y]].dropna()
    print(x, "vs", y)
    print(" Pearson:", stats.pearsonr(tmp[x], tmp[y]))
    print(" Spearman:", stats.spearmanr(tmp[x], tmp[y]))

ap = df.loc[df["aprobo"].eq(1), "promedio_evaluaciones"].dropna()
no = df.loc[df["aprobo"].eq(0), "promedio_evaluaciones"].dropna()
print("t-test Welch:", stats.ttest_ind(ap, no, equal_var=False))

anova_groups = [g["promedio_evaluaciones"].dropna().to_numpy() for _, g in df.groupby("highest_education")]
print("ANOVA highest_education:", stats.f_oneway(*anova_groups))

chi = pd.crosstab(df["gender"], df["final_result"])
print("Chi-cuadrado gender x final_result:", stats.chi2_contingency(chi)[:3])

if sm is not None:
    ancova_data = df[["promedio_evaluaciones", "gender", "clicks_28d", "studied_credits"]].dropna()
    modelo = ols("promedio_evaluaciones ~ C(gender) + clicks_28d + studied_credits", data=ancova_data).fit()
    display(sm.stats.anova_lm(modelo, typ=2))
    manova_data = df[["aprobo", "clicks_28d", "dias_activos_28d", "promedio_evaluaciones"]].dropna()
    print(MANOVA.from_formula("clicks_28d + dias_activos_28d + promedio_evaluaciones ~ aprobo", data=manova_data).mv_test())

## 6. Machine learning: RandomForestClassifier y RandomForestRegressor

In [ ]:
features = ["clicks_28d", "dias_activos_28d", "studied_credits", "num_of_prev_attempts",
            "educacion_ordinal", "edad_ordinal", "gender", "disability", "code_module"]
numeric = ["clicks_28d", "dias_activos_28d", "studied_credits", "num_of_prev_attempts", "educacion_ordinal", "edad_ordinal"]
categorical = ["gender", "disability", "code_module"]

pre = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), categorical),
])

model_clf = Pipeline([
    ("pre", pre),
    ("rf", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, class_weight="balanced", n_jobs=-1)),
])
model_reg = Pipeline([
    ("pre", pre),
    ("rf", RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)),
])

clf_data = df[features + ["aprobo"]].dropna(subset=["aprobo"])
X_train, X_test, y_train, y_test = train_test_split(
    clf_data[features], clf_data["aprobo"], test_size=0.25, random_state=RANDOM_STATE, stratify=clf_data["aprobo"]
)
model_clf.fit(X_train, y_train)
pred = model_clf.predict(X_test)
print(classification_report(y_test, pred))
ConfusionMatrixDisplay.from_predictions(y_test, pred)
plt.title("Matriz de confusión - RandomForestClassifier")
plt.show()

reg_data = df[features + ["promedio_evaluaciones"]].dropna(subset=["promedio_evaluaciones"])
X_train, X_test, y_train, y_test = train_test_split(
    reg_data[features], reg_data["promedio_evaluaciones"], test_size=0.25, random_state=RANDOM_STATE
)
model_reg.fit(X_train, y_train)
pred_reg = model_reg.predict(X_test)
print("MAE:", mean_absolute_error(y_test, pred_reg))
print("R2:", r2_score(y_test, pred_reg))

## 7. Conclusión

Los indicadores tempranos de interacción en el VLE y el desempeño en evaluaciones deben interpretarse conjuntamente. Los modelos predictivos son útiles como línea base, pero no sustituyen el análisis pedagógico ni la validación institucional.